## Módulo 2 - Review Attributes

### Introducción

En este módulo construimos la dimensión estructurada del sistema de recomendación: una representación tabular donde cada hotel queda caracterizado por un conjunto de atributos relevantes (wifi, limpieza, ubicación, desayuno, ruido, vista, personal, entre otros). Esta capa complementa al motor de búsqueda semántica del módulo 3, ya que permite resolver consultas de naturaleza distinta: aspectos discretos y verificables ("¿tiene buen wifi?", "¿el desayuno es valorado?") que se prestan mejor a un filtrado duro por columnas que a una búsqueda por similitud vectorial.

Para extraer estos atributos a partir de las reviews necesitamos una tarea distinta a la del módulo 3. Mientras que los embeddings comprimen el significado global de una review en un vector, lo que buscamos acá es leer el texto, identificar menciones a categorías predefinidas, y asignar un sentimiento a cada una. Es una tarea de comprensión y generación estructurada que un encoder no puede resolver ya que requiere razonar sobre el texto y producir una salida tabular nueva. Por eso usamos un LLM generativo: le pasamos al modelo el comentario positivo y negativo de cada review junto con un prompt detallado que define las categorías a evaluar, y obtenemos como respuesta un JSON estructurado con un valor de sentimiento (1, -1 o 0) por categoría. Este enfoque resuelve la extracción zero-shot, sin necesidad de etiquetar miles de reviews manualmente para entrenar un clasificador supervisado.

Las categorías de análisis se definen en un archivo YAML externo (config_features.yaml), no se hardcodean en el código. Esto permite agregar, eliminar o redefinir categorías sin tocar la lógica del pipeline, y el prompt del LLM se construye dinámicamente a partir de esa configuración, por lo que cualquier modificación del esquema se propaga automáticamente al resto del procesamiento.
El procesamiento masivo de cientos de miles de reviews implica además decisiones de ingeniería que afectan tiempo y costo. En lugar de consumir una API paga, levantamos el modelo Qwen/Qwen2.5-3B-Instruct en un contenedor Docker local utilizando vLLM, un servidor de inferencia que aporta optimizaciones de bajo nivel (continuous batching, KV caching, prefix caching) para maximizar el throughput sobre la GPU disponible. Las reviews se procesan en batches concurrentes mediante un cliente asincrónico, y el pipeline incorpora retries automáticos, timeouts, logs de fallos y checkpoints incrementales que permiten retomar el procesamiento desde el último punto válido sin reprocesar el dataset completo en caso de interrupción.

Como salida, el módulo produce dos artefactos que el resto del sistema va a consumir. El primero es una tabla a nivel review con una columna por categoría (output_reviews_completo.csv) y una tabla a nivel hotel con la suma de sentimientos por categoría más la columna destino (output_hotel_scores.csv). Esta última es la que alimentará el motor híbrido del módulo 4. Adicionalmente se construye una base SQLite (hoteles.db) con la información agregada, que permite formular consultas estructuradas mediante SQL como mecanismo de filtrado por atributos en el sistema final.

In [ ]:
# Importamos librerías
import pandas as pd
import json
import yaml
from tqdm import tqdm
import asyncio
from openai import AsyncOpenAI
from pathlib import Path
import sqlite3
import numpy as np

In [ ]:
# Definimos todas las rutas de archivos que va usar el notebook
current_path = Path.cwd()

# Lista
FEATURE_CONFIG = current_path/"config_features.YAML"
DATA = current_path.parent/"Data"/"Final"/"eda_final_dataset.csv"
CHECKPOINT = current_path.parent/"Outputs"/"checkpoint.csv"
FAILED_LOG = current_path.parent/"Outputs"/"log.txt"
OUTPUT_EJEMPLO = current_path.parent/"Outputs"/"output_ejemplo.csv"
OUTPUT = current_path.parent/"Outputs"/"output.csv"
OUTPUT_PROCESAMIENTO_REVIEWS = current_path.parent/"Outputs"/"output_reviews_completo.csv"
OUTPUT_HOTEL_SCORES = current_path.parent/"Outputs"/"output_hotel_scores.csv"
BASE_DE_DATOS = current_path.parent/"Data"/"hoteles.db"

### Carga Inicial (Dataset y Categorías)

Definimos las categorías de análisis en un archivo YAML para que la estructura del pipeline sea configurable y fácil de mantener. Esto nos permite agregar, eliminar o modificar categorías y descripciones sin cambiar el código principal, facilitando iteraciones rápidas sobre el esquema de clasificación de reseñas hoteleras.

Luego cargamos el dataset de trabajo, que contiene las reviews ya procesadas en la etapa de EDA.

In [ ]:
# Cargamos el archivo YAML con la configuración de categorías
with open(FEATURE_CONFIG, "r", encoding = "utf-8") as f:
    config = yaml.safe_load(f)

# Extraemos la lista de categorías que vamos a usar para estructurar las reviews
categorias = config["categorias"]

# Cargamos el dataset de trabajo con las reviews de hoteles
df_work = pd.read_csv(DATA)

/var/folders/3_/n3rxgyx52s38d1zrs4t2d3v00000gn/T/ipykernel_77631/4020393184.py:9: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_work = pd.read_csv(DATA)


In [ ]:
# Head
df_work.head(10)

,hotel_id_review,fecha_review,fecha_checkout,idioma,overall_rating,score_ubicacion,score_limpieza,score_habitacion,score_servicio,score_personal,...,name,city,country,star_rating,hotel_type,amenities,meal_plans,len_positivo,len_negativo,destino
0,4669631,"10 febrero, 2025","31 enero, 2025",ES,95.56,100,100,100,100,100,...,Buona Vitta Gramado Resort & Spa by Gramado Parks,Gramado,Brasil,5.0,Hoteles,"{'Gimnasio','Valet parking','Servicio de spa',...",{'CONTINENTAL_BREAKFAST'},54,0,Brasil - Gramado
1,231021,"10 febrero, 2025","6 febrero, 2025",ES,100.00,100,100,100,100,100,...,Majestic Rio Palace Hotel,Rio De Janeiro,Brasil,3.0,Hoteles,{'Baño adaptado para personas con movilidad re...,{'BUFFET_BREAKFAST'},330,0,Brasil - Rio De Janeiro
2,265830,"10 febrero, 2025","4 febrero, 2025",ES,100.00,100,100,100,100,100,...,Melia Casa Maya Cancun All Inclusive,Cancún,México,4.0,Hoteles,"{'Terraza','Periodo de desinfección entre esta...",{'ALL_INCLUSIVE'},49,0,México - Cancún
3,358839,"10 febrero, 2025","9 febrero, 2025",ES,91.11,80,100,100,100,100,...,Gran Hotel Continental,Mar Del Plata,Argentina,3.0,Hoteles,"{'Sala de reuniones','Bar','Servicio de conser...",{'BUFFET_BREAKFAST'},71,49,Argentina - Mar Del Plata
4,999361,"10 febrero, 2025","10 febrero, 2025",ES,82.22,100,100,60,100,100,...,Ibis Budget Rio de Janeiro - Praia de Botafogo,Rio De Janeiro,Brasil,2.0,Hoteles,"{'Recepción 24 hrs','Seguridad 24 hrs','Restau...","{'BREAKFAST','BUFFET_BREAKFAST','ROOM_ONLY'}",12,7,Brasil - Rio De Janeiro
5,469657,"10 febrero, 2025","31 enero, 2025",PT,80.00,100,40,100,60,60,...,Universal Cabana Bay Beach Resort,Orlando,Estados Unidos,3.0,Hoteles,"{'Piscina','Recepción 24 hrs','Parque acuático...",NaN,102,92,Estados Unidos - Orlando
6,285508,"10 febrero, 2025","8 febrero, 2025",ES,66.67,80,80,60,60,80,...,Catussaba Resort Hotel,Salvador,Brasil,4.0,Resorts,"{'Toallas','Servicio de lavandería con costo a...","{'BUFFET_BREAKFAST','FULL_BOARD','HALF_BOARD_A...",443,184,Brasil - Salvador
7,245305,"10 febrero, 2025","9 febrero, 2025",ES,84.44,100,80,80,100,100,...,Colonna Park Hotel,Búzios,Brasil,4.0,Hoteles,"{'Restaurante','Servicio a la habitación','Pis...",{'BUFFET_BREAKFAST'},48,38,Brasil - Búzios
8,205656,"10 febrero, 2025","7 febrero, 2025",PT,91.11,100,100,100,100,80,...,Hotel Praia Centro,Fortaleza,Brasil,4.0,Hoteles,"{'Gimnasio','Servicio a la habitación','Tensió...",{'BUFFET_BREAKFAST'},58,59,Brasil - Fortaleza
9,249616,"10 febrero, 2025","3 febrero, 2025",ES,55.56,60,60,40,60,60,...,Krystal Beach Acapulco,Acapulco,México,4.0,Hoteles,"{'Sillas de playa','Desayuno con costo adicion...","{'ALL_INCLUSIVE','BUFFET_BREAKFAST','ROOM_ONLY'}",6,51,México - Acapulco


In [ ]:
# Describe (estadísticas de las variables numéricas)
df_work.describe()

,hotel_id_review,overall_rating,score_ubicacion,score_limpieza,score_habitacion,score_servicio,score_personal,score_banio,score_internet,score_comida,...,es_pareja,es_familia,es_viaje_negocios,es_perfil_lujo,es_perfil_economico,es_romantico,es_perfil_relax,star_rating,len_positivo,len_negativo
count,2.513700e+05,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,...,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000,251370.000000
mean,9.869396e+05,83.958647,90.213629,85.042607,82.059355,83.654454,88.214266,80.950631,78.994709,83.609261,...,0.372324,0.766870,0.264395,0.126853,0.377929,0.318952,0.511449,3.625433,72.500637,89.686649
std,1.446333e+06,16.373437,15.794884,20.917628,21.465986,21.034073,19.148001,21.811989,22.505057,20.739165,...,0.483425,0.422825,0.441011,0.332809,0.484871,0.466071,0.499870,0.922347,99.539513,146.796852
min,2.000310e+05,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.854440e+05,77.780000,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000,60.000000,80.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,20.000000,8.000000
50%,3.519070e+05,88.890000,100.000000,100.000000,80.000000,100.000000,100.000000,80.000000,80.000000,100.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,4.000000,41.000000,40.000000
75%,9.266220e+05,97.780000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,...,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,4.000000,86.000000,107.000000
max,6.818082e+06,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000,1416.000000,1412.000000


### Construcción Dinámica Del Prompt Estructurador

En este paso construimos el prompt que va a usar el LLM para transformar cada reseña hotelera en una salida estructurada por categorías.

La idea es que, a partir de los comentarios positivos y negativos de cada review, el modelo identifique si se mencionan aspectos específicos del hotel, como limpieza, ubicación, personal, desayuno, wifi, vista u otras categorías definidas en el archivo de configuración.

Para cada categoría, el modelo debe asignar un valor de sentimiento:

- `1` si la mención es positiva.
- `-1` si la mención es negativa.
- `0` si no hay evidencia suficiente sobre esa categoría.

Un punto importante es que no hardcodeamos las categorías dentro del prompt. En cambio, las tomamos directamente desde el archivo YAML cargado en el paso anterior. Esto permite modificar el esquema de análisis (agregando, eliminando o ajustando categorías) sin cambiar la lógica principal del pipeline.

Además, diseñamos el prompt para contemplar casos reales de reviews, como comentarios negativos escritos dentro del campo positivo, listas cortas de atributos o menciones ambiguas. También forzamos al modelo a responder únicamente con un JSON válido, porque ese formato es necesario para procesar automáticamente los resultados en las siguientes etapas.

Toda la definición de categorías y sus descripciones se construye dinámicamente a partir del archivo YAML de configuración. De esta manera, cualquier modificación en categorías, reglas o definiciones puede realizarse sin alterar el código principal del pipeline.

In [ ]:
# -----------------------------------
# Construcción automática
# -----------------------------------

categorias_texto = "\n".join(
    [
        f'- {cat["nombre"]}: {cat["descripcion"]}'
        for cat in categorias
    ]
)


json_ejemplo = {
    cat["nombre"]: "0"
    for cat in categorias
}

# Opcionalmente modificamos algunas
if "wifi" in json_ejemplo:
    json_ejemplo["wifi"] = "-1"

# Opcionalmente modificamos algunas
if "limpieza" in json_ejemplo:
    json_ejemplo["limpieza"] = "-1"

if "vista" in json_ejemplo:
    json_ejemplo["vista"] = "1"

# -----------------------------------
# Prompt final
# -----------------------------------

SYSTEM_PROMPT_ESTRUCTURADOR_V2 = f"""
Sos un sistema de extracción de sentimientos de reviews hoteleras. Cada review esta formada por un comentario positivo y un comentario negativo.

Categorias posibles:
{categorias_texto}

Para cada categoria devolver SOLO uno de estos valores:
- 1 (positivo)
- -1 (negativo)
- 0 (neutro)

Reglas IMPORTANTES:
- Responder SOLO un JSON válido
- No agregar explicaciones
- No agregar markdown
- No agregar texto extra
- Incluir TODAS las categorias en el JSON
- Si una categoria no se menciona directa o implicitamente devolver "0"
- Si un comentario positivo o negativo enumera elementos, asumir que todos los elementos comparten el mismo sentimiento
- Cada categoría debe evaluarse independientemente. Si el texto contiene cualquier mención directa o indirecta razonable de un aspecto (aunque sea dentro de una frase secundaria, enumeración o detalle adicional), se debe asignar sentimiento positivo (1) o negativo (-1) según el contexto. Solo se asigna 0 si no hay evidencia textual suficiente sobre la categoría.

Ejemplo:
Comentario Positivo: "el hotel era muy lindo con vista a la playa."
Comentario Negativo: "wifi,limpieza"

Salida esperada:
{json.dumps(json_ejemplo, indent = 2, ensure_ascii = False)}


Consideraciones importantes:
1) El usuario pudo haber escrito toda su reseña en uno de ambos comentarios. Por ejemplo, un huesped pudo haber incluido comentarios negativos en el campo de positivos, y si ese es el caso no debemos considerarlo como positivo si tiene connotacion negativa (o visceversa)
2) El usuario pudo haber dejado un comentario negativo y/o positivo solo listando cosas. En esos casos, debemos interpretar que los califica como malos o buenos a todos ellos respectivamente.
3) Si no se proporcionan detalles buenos o malos, no debe asumirse nada

"""

print(SYSTEM_PROMPT_ESTRUCTURADOR_V2)


Sos un sistema de extracción de sentimientos de reviews hoteleras. Cada review esta formada por un comentario positivo y un comentario negativo.

Categorias posibles:
- wifi: comentarios sobre wifi, internet, conexión, señal o velocidad de internet
- vista: comentarios sobre paisajes, panoramas o visuales
- ubicacion: comentarios sobre localización, cercanía o accesibilidad
- ruido: comentarios sobre sonidos, música, aislamiento o tranquilidad
- limpieza: comentarios sobre higiene, aseo u orden
- transporte: comentarios sobre transporte público, accesos o movilidad
- personal: comentarios sobre atención, amabilidad o servicio del personal
- desayuno: comentarios sobre desayuno o cafetería matutina
- restaurante: comentarios sobre comida, restaurante, menú, almuerzo o cena
- habitacion: comentarios sobre habitación, cuarto, cama, baño, muebles, tamaño, comodidad o estado de la habitación
- aire_acondicionado: comentarios sobre temperatura, aire acondicionado o ventilación
- estacionami

### Configuración Del Servidor De Inferencia Con vLLM

Inicialmente, el pipeline realizaba dos inferencias separadas por cada reseña. En una primera etapa, se utilizaba un prompt para interpretar el comentario positivo y el comentario negativo, y consolidarlos en una única reseña integrada. Luego, sobre ese texto unificado, se ejecutaba una segunda inferencia encargada de extraer los sentimientos por categoría.

Si bien este enfoque funcionaba, duplicaba la cantidad de llamadas al modelo. Esto aumentaba los tiempos de procesamiento y el costo computacional total, especialmente considerando que el dataset contiene un volumen alto de reseñas.

Por ese motivo, rediseñamos la estrategia de prompting para que el modelo reciba simultáneamente el comentario positivo y el comentario negativo, y realice directamente la extracción estructurada de sentimientos en una sola inferencia. De esta manera, simplificamos el pipeline y eliminamos una etapa completa de procesamiento por reseña.

Una vez optimizada la lógica del prompt, el siguiente cuello de botella pasó a ser la velocidad de inferencia. Para mejorar el throughput, decidimos utilizar vLLM como servidor local de inferencia. Levantamos el modelo dentro de un contenedor Docker, usando la GPU local y exponiendo una API compatible con OpenAI.

En este caso, servimos el modelo `Qwen/Qwen2.5-3B-Instruct`, que luego consumimos desde Python mediante un cliente asincrónico. Esto nos permite enviar múltiples requests en paralelo y aprovechar mejor la capacidad de procesamiento disponible.

vLLM incorpora optimizaciones útiles para este escenario, como:

* Continuous batching
* KV caching
* Prefix caching
* Async scheduling
* Ejecución tensorial acelerada en GPU

Estas optimizaciones mejoran significativamente el throughput y reducen la latencia, especialmente al procesar grandes volúmenes de reseñas mediante batches concurrentes.

Comando utilizado:

```bash
docker run --gpus all -p 8000:8000 \
  vllm/vllm-openai:latest \
  --model Qwen/Qwen2.5-3B-Instruct
```


In [ ]:
# Creamos un cliente asincrónico compatible con la API de OpenAI
# En lugar de apuntar a OpenAI, lo conectamos al servidor local de vLLM
client = AsyncOpenAI(
    base_url = "http://localhost:8000/v1",
    api_key = "dummy"
)

# Definimos una función asincrónica para estructurar una reseña individual
async def call(row):
    return await client.chat.completions.create(
        model = "Qwen/Qwen2.5-3B-Instruct",
        messages = [
            # Usamos el prompt estructurador definido previamente
            {
                "role": "system",
                "content": SYSTEM_PROMPT_ESTRUCTURADOR_V2
            },
            # Enviamos al modelo el comentario positivo y negativo de la review
            {
                "role": "user",
                "content": f"""
Pos: {row['texto_positivo']}
Neg: {row['texto_negativo']}
"""
            }
        ],
        # Fijamos temperatura 0 para buscar respuestas más consistentes y determinísticas
        temperature = 0
    )

# Definimos una función para ejecutar varias inferencias en paralelo
async def run(rows):
    return await asyncio.gather(
        *[call(r) for r in rows]
    )

Antes de aplicar el pipeline sobre todo el dataset, probamos la inferencia con un subconjunto pequeño de reviews. Esto nos permite validar que el servidor de vLLM responda correctamente, que el prompt devuelva un JSON válido y que la estructura de salida pueda parsearse sin errores antes de escalar el procesamiento.

In [ ]:
# Tomamos una muestra inicial de reviews para validar el pipeline
rows = df_work.head(100).to_dict("records")

# Ejecutamos la inferencia asincrónica sobre la muestra
results = await run(rows)

# Guardamos las respuestas crudas generadas por el modelo
outputs = []

for r, res in zip(rows, results):
    # Extraemos el contenido textual de la respuesta del modelo
    text = res.choices[0].message.content
    outputs.append(text)

# Parseamos las respuestas como JSON para validar que tengan una estructura procesable
parsed = [json.loads(o) for o in outputs]

Una vez que validamos que el modelo devuelve un JSON válido por cada reseña, transformamos esas respuestas en una tabla estructurada.

Para cada review, conservamos el `hotel_id_review` y unimos el comentario positivo y negativo en una columna de referencia. Luego recorremos todas las categorías definidas en el YAML y guardamos el sentimiento asignado por el modelo para cada una de ellas.

El resultado final es un DataFrame donde cada fila representa una reseña y cada columna de categoría contiene un valor numérico: `1` para sentimiento positivo, `-1` para sentimiento negativo y `0` cuando no hay evidencia suficiente.

In [ ]:
# Creamos una lista donde vamos a guardar una fila estructurada por cada review
resultados = []

# Recorremos en paralelo las reviews originales y los sentimientos parseados del modelo
for row, sentiments in zip(rows, parsed):

    # Recuperamos el identificador de la review/hotel
    hotel_id = row["hotel_id_review"]

    # Recuperamos los comentarios originales para mantener trazabilidad
    comentario_positivo = row["texto_positivo"]
    comentario_negativo = row["texto_negativo"]

    # Construimos la fila base con el identificador y los comentarios originales
    fila = {
        "hotel_id_review": hotel_id,
        "Comentarios": f"Comentario pos = {comentario_positivo}. Comentario neg = {comentario_negativo}"
    }   

    # Agregamos una columna por cada categoría definida en el YAML
    for categoria in categorias:

        nombre = categoria["nombre"]

        # Guardamos el sentimiento detectado por el modelo
        # Si la categoría no aparece en la respuesta, asumimos 0 por seguridad
        fila[nombre] = int(
            sentiments.get(nombre, 0)
        )

    # Sumamos la fila estructurada al resultado final
    resultados.append(fila)

# Convertimos la lista de resultados en un DataFrame tabular
df_resultados_v2_api = pd.DataFrame(resultados)

In [ ]:
# Mostramos texto completo en columnas
pd.set_option("display.max_colwidth", None)

# Evitamos cortes por ancho
pd.set_option("display.width", None)

In [ ]:
# Mostramos resultados
df_resultados_v2_api

,hotel_id_review,Comentarios,wifi,vista,ubicacion,ruido,limpieza,transporte,personal,desayuno,restaurante,habitacion,aire_acondicionado,estacionamiento,seguridad,checkin
0,4669631,"Comentario pos=El gran espacio que dispone, el restaurante muy bueno.. Comentario neg=nan",0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,231021,"Comentario pos=La atención del personal es espectacular, son muy simpáticos y te ayudan desde el primer momento a ubicarte en la ciudad. Ademas, su ubicación te permite llegar a la playa en menos de 10 minutos, cuenta con farmacias, restoranes, un banco y supermercados cerca. Volvería a alojarme en el hotel sin pensarlo, muy buena experiencia.. Comentario neg=nan",0,0,1,0,0,0,1,0,0,0,0,0,0,0
2,265830,"Comentario pos=Ubicación,la playa y las instalaciones en general. Comentario neg=nan",0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,358839,Comentario pos=Excelente el personal. La habitación triple superior en calidad óptima.. Comentario neg=Desayuno muy bueno pero le faltarían más opciones,0,0,0,0,0,0,1,-1,0,1,0,0,0,0
4,999361,Comentario pos=Localização.. Comentario neg=Preços.,0,0,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,240698,"Comentario pos=Fui con mi mamá y mi hijo de 3 años por una noche . La verdad la atención del personal es excelente reserve una habitación y tenía cama grande . El señor me preguntó si estaba bien o si quería 2 camas ya que éramos 3 . Cambie de habitación y el señor me llamo para preguntar si estábamos cómodos . Habitación chica pero más que cómoda y linda . Baño también super limpio, comodo .El desayuno un 10 tenés de todo creo q es lo mejor .No tengo quejas. Comentario neg=Una mini queja que es mínima el enchufe del baño no andaba bien para utilizar mi plancha de pelo y solo hay uno .",1,0,0,0,1,0,1,1,0,1,0,0,0,0
96,285432,"Comentario pos=A localização é excelente, de frente para praia do porto da barra. Comentario neg=O hotel é ruim e desorganizado, pelo que me parece, não tem gerência. O quarto é sujo e trocam roupas de cama e banho somente a cada 2 dias, além do cartão de acesso do quarto que vc precisa ficar recarregando na recepção. A piscina é boa mas também não tem toalha para secar quando sair, vc anda pelo hotel todo molhado mesmo… é uma pena pois com uma localização tão boa, se fosse mais organizado, seria muito mais recomendado. Com certeza não voltaremos a este hotel.",0,1,1,0,-1,0,-1,0,0,-1,0,0,0,0
97,4738626,Comentario pos=El aseo. Comentario neg=Sala de juegos muy básica,0,0,0,0,1,0,0,0,0,0,0,0,0,0
98,688026,"Comentario pos=Me gusto el desayuno, la merienda, la atención a los huéspedes, los detalles. Comentario neg=nan",0,0,0,0,0,0,1,1,0,0,0,0,0,0


In [ ]:
# Guardamos los resultados
df_resultados_v2_api.to_csv(OUTPUT_EJEMPLO, index = False)

### Procesamiento Masivo Con Retries y Control De Fallos

Creamos además una pequeña función wrapper encargada de ejecutar las llamadas al modelo de forma robusta y tolerante a fallos. Esta función incorpora un timeout configurable por batch y una lógica de retry automático para evitar que el pipeline quede bloqueado indefinidamente ante problemas puntuales durante la inferencia, como respuestas colgadas, outputs inválidos o requests extremadamente lentas.

Para cada batch, la función intenta ejecutar la inferencia múltiples veces antes de marcarlo como fallido. En caso de timeout o error, se registra el incidente en consola y se espera unos segundos antes de reintentar la ejecución. Si luego de todos los intentos el batch continúa fallando, este se registra automáticamente en un archivo de logs (FAILED_LOG) para su posterior análisis o reprocesamiento manual.

Este enfoque permite que el procesamiento continúe incluso frente a errores aislados del modelo o del servidor de inferencia, evitando perder horas de ejecución por un pequeño conjunto de reviews problemáticas y haciendo el pipeline considerablemente más resiliente y estable para procesamiento masivo.

In [ ]:
# Definimos parámetros generales para controlar la tolerancia a fallos del procesamiento
MAX_RETRIES = 2
TIME_OUT = 30
RETRY_WAIT = 5

# Creamos una función wrapper para ejecutar un batch con timeout y reintentos
async def run_with_retry(
    rows,
    batch_num,
    timeout = TIME_OUT,
    max_retries = MAX_RETRIES,
    retry_wait = RETRY_WAIT
):

    # Intentamos procesar el batch hasta alcanzar el máximo de reintentos definido
    for attempt in range(1, max_retries + 1):

        try:

            print(
                f"Batch {batch_num} - intento {attempt}"
            )

            # Ejecutamos la inferencia del batch con un límite máximo de tiempo
            results = await asyncio.wait_for(
                run(rows),
                timeout = timeout
            )

            print(
                f"Batch {batch_num} completado"
            )

            # Si el batch se completa correctamente, devolvemos los resultados
            return results

        except asyncio.TimeoutError:

            # Registramos si el batch superó el tiempo máximo permitido
            print(
                f"Timeout en batch {batch_num} "
                f"(intento {attempt}/{max_retries})"
            )

        except Exception as e:

            # Registramos cualquier otro error inesperado durante la inferencia
            print(
                f"Error en batch {batch_num}: {e} "
                f"(intento {attempt}/{max_retries})"
            )

        # Esperamos unos segundos antes de volver a intentar
        await asyncio.sleep(retry_wait)

    # Si agotamos todos los intentos, marcamos el batch como fallido
    print(
        f"Batch {batch_num} falló definitivamente"
    )

    # Guardamos el batch fallido en un archivo de logs para revisarlo luego
    with open(FAILED_LOG, "a", encoding = "utf-8") as f:

        f.write(
            f"Batch {batch_num} falló\n"
        )

    # Devolvemos None para indicar que no obtuvimos resultados válidos
    return None

Una vez definida la función robusta de inferencia, aplicamos el pipeline sobre todo el dataset de trabajo.

In [ ]:
# Definimos el tamaño de cada batch de procesamiento
BATCH_SIZE = 100

# Definimos cada cuántos batches vamos a guardar un checkpoint parcial
CHECKPOINT_EVERY = 10 # Guarda cada 10 batches

# Inicializamos la lista donde vamos a acumular todas las filas procesadas
resultados_totales = []

# Definimos desde qué fila queremos comenzar el procesamiento
# Esto nos permite retomar manualmente desde cierto punto si fuera necesario
filas_procesadas = 0

# Inicializamos un contador de batches fallidos
failed_batches_count = 0

In [ ]:
# Recorremos el dataset completo en batches
for batch_num, i in enumerate(
    tqdm(
        range(
            filas_procesadas,
            len(df_work),
            BATCH_SIZE
        ),
        desc = "Procesando batches"
    ),
    start = (filas_procesadas // BATCH_SIZE) + 1 + failed_batches_count
):

    # Tomamos el bloque de filas correspondiente al batch actual
    batch_df = df_work.iloc[i:i+BATCH_SIZE]

    # Convertimos el batch a lista de diccionarios para enviarlo al modelo
    rows = batch_df.to_dict("records")

    # Ejecutamos la inferencia usando la función con retries y timeout
    results = await run_with_retry(
        rows,
        batch_num
    )

    # Si el batch falló definitivamente, lo salteamos y seguimos con el próximo
    if results is None:
        continue
    
    # Guardamos las respuestas crudas del modelo
    outputs = []

    for res in results:

        # Extraemos el contenido textual de cada respuesta
        text = res.choices[0].message.content
        outputs.append(text)

    # Parseamos las respuestas del modelo a JSON de forma robusta
    parsed = []

    for o in outputs:

        try:

            # Intentamos convertir la respuesta del modelo en diccionario
            parsed.append(json.loads(o))

        except Exception as e:

            # Si la respuesta no es un JSON válido, registramos el error
            print(f"Error parseando JSON: {e}")

            # Agregamos un diccionario vacío para no cortar el pipeline
            parsed.append({})

    # Construimos las filas estructuradas a partir de las reviews y los sentimientos parseados
    for row, sentiments in zip(rows, parsed):

        # Creamos la fila base con identificador y comentarios originales
        fila = {
            "hotel_id_review": row["hotel_id_review"],
            "Comentarios": (
                f"Comentario pos={row['texto_positivo']}. "
                f"Comentario neg={row['texto_negativo']}"
            )
        }

        # Recorremos todas las categorías definidas en el YAML
        for categoria in categorias:

            nombre = categoria["nombre"]

            # Recuperamos el valor asignado por el modelo
            # Si la categoría no aparece, usamos 0 como valor seguro
            valor = sentiments.get(nombre, 0)

            try:

                # Validamos que el valor recibido sea convertible a entero
                if not isinstance(valor, (int, float, str)):
                    raise ValueError(
                        f"Tipo inválido recibido: {type(valor)}"
                    )

                # Guardamos el sentimiento como entero: 1, -1 o 0
                fila[nombre] = int(valor)

            except Exception as e:

                # Si el valor no se puede parsear, registramos el problema
                print(
                    f"Error parseando categoria '{nombre}' "
                    f"en batch {batch_num}: {e}"
                )

                # Guardamos el output problemático para revisarlo posteriormente
                with open(
                    "bad_outputs.txt",
                    "a",
                    encoding = "utf-8"
                ) as f:

                    f.write(
                        f"\n---\n"
                        f"Batch: {batch_num}\n"
                        f"Categoria: {nombre}\n"
                        f"Valor recibido: {valor}\n"
                        f"Output completo:\n"
                        f"{json.dumps(sentiments, ensure_ascii = False)}\n"
                    )

                # Asignamos 0 como fallback para no interrumpir el procesamiento
                fila[nombre] = 0

        # Agregamos la fila procesada al acumulado total
        resultados_totales.append(fila)

    # Mostramos el avance general del procesamiento
    print(
        f"Procesados: {i + len(batch_df)} / {len(df_work)}"
    )

    # Guardamos un checkpoint incremental cada cierta cantidad de batches
    if batch_num % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(resultados_totales)

        checkpoint_df.to_csv(
            CHECKPOINT,
            index = False
        )

        print(
            f"Checkpoint guardado en batch {batch_num}"
        )

Implementamos además un pequeño bloque de recuperación que permite reanudar automáticamente el procesamiento en caso de fallos durante la ejecución. Este mecanismo carga el último checkpoint persistido, reconstruye en memoria los resultados ya procesados y calcula cuántas filas fueron completadas correctamente. Adicionalmente, se leen los batches fallidos registrados en el log para mantener consistente la numeración y trazabilidad del pipeline.

De esta manera, si ocurre un error o interrupción del proceso, simplemente es posible volver a ejecutar el bloque principal de procesamiento y continuar prácticamente desde el punto exacto donde se detuvo, evitando reprocesar cientos de miles de reseñas innecesariamente y reduciendo considerablemente el tiempo de recuperación.

In [ ]:
# Cargamos el checkpoint con los resultados ya procesados
checkpoint_df = pd.read_csv(CHECKPOINT)

# Calculamos cuántas filas ya fueron procesadas correctamente
filas_procesadas = len(checkpoint_df)

# Reconstruimos la lista acumulada de resultados a partir del checkpoint
resultados_totales = (
    checkpoint_df.to_dict("records")
)

# Intentamos recuperar la cantidad de batches fallidos registrados previamente
try:

    with open(
        FAILED_LOG,
        "r",
        encoding = "utf-8"
    ) as f:

        # Contamos la cantidad de líneas del log para estimar los batches fallidos
        failed_batches_count = len(
            f.readlines()
        )

except FileNotFoundError:

    # Si no existe log de fallos, asumimos que no hay batches fallidos registrados
    failed_batches_count = 0

Una vez finalizado el procesamiento principal, cargamos los resultados generados y realizamos una validación de completitud para identificar qué reseñas no lograron procesarse correctamente. Debido a que el pipeline trabaja en batches y posee mecanismos de tolerancia a fallos —como timeouts, retries automáticos y continuación del procesamiento frente a errores aislados— puede ocurrir que algunos batches específicos fallen definitivamente y sean omitidos temporalmente para evitar detener toda la ejecución.

Para detectar estos casos, reconstruimos exactamente las claves utilizadas durante el procesamiento (hotel_id_review + Comentarios) y las contrastamos contra el dataset original. Esto nos permite identificar con precisión qué filas no llegaron al resultado final, incluso en escenarios donde existen IDs repetidos o múltiples reviews asociadas al mismo hotel.

Una vez aisladas las reseñas faltantes, generamos un nuevo dataframe únicamente con esos registros y ejecutamos un segundo procesamiento focalizado sobre ese subconjunto reducido. En esta segunda pasada, las reviews restantes se procesan utilizando batches considerablemente más pequeños, reduciendo la probabilidad de timeouts o fallos asociados a prompts problemáticos, outputs inesperados o reseñas particularmente largas.

Dado que los registros faltantes representan menos del 1% del total de los datos, cualquier error residual en esta etapa ya tiene un impacto estadísticamente despreciable sobre el dataset final. Este enfoque permite maximizar el throughput del procesamiento principal sin comprometer la robustez general del pipeline ni requerir reprocesar cientos de miles de reseñas completas.

In [ ]:
# Resultados
resultados_pre = pd.read_csv(OUTPUT)

In [ ]:
# Reconstruimos las mismas claves de comparación que usamos durante el procesamiento
rows_originales = df_work.to_dict("records")

# Armamos un DataFrame auxiliar con el identificador y los comentarios originales
df_original_compare = pd.DataFrame([
    {
        "hotel_id_review": row["hotel_id_review"],
        "Comentarios": (
            f"Comentario pos={row['texto_positivo']}. "
            f"Comentario neg={row['texto_negativo']}"
        )
    }
    for row in rows_originales
])

# Agregamos el índice original para poder recuperar luego las filas exactas desde df_work
df_original_compare["_idx"] = df_work.index

# Comparamos el dataset original contra los resultados ya procesados usando ambas claves
faltantes_idx = df_original_compare.merge(
    resultados_pre,
    on = ["hotel_id_review", "Comentarios"],
    how = "left",
    indicator = True
)

# Nos quedamos únicamente con las filas que están en el dataset original, pero no aparecen en el resultado procesado
faltantes_idx = faltantes_idx[
    faltantes_idx["_merge"] == "left_only"
]

# Recuperamos desde df_work las filas originales exactas que quedaron faltantes
df_faltantes_full = df_work.loc[
    faltantes_idx["_idx"]
].copy()

# Mostramos cuántas filas quedaron pendientes para reprocesar
print(
    f"Filas recuperadas para reprocesar: "
    f"{len(df_faltantes_full)}"
)

# Revisamos las primeras filas faltantes
df_faltantes_full.head()

Filas recuperadas para reprocesar: 1192


,hotel_id_review,fecha_review,fecha_checkout,idioma,overall_rating,score_ubicacion,score_limpieza,score_habitacion,score_servicio,score_personal,...,name,city,country,star_rating,hotel_type,amenities,meal_plans,len_positivo,len_negativo,destino
23200,204912,"8 febrero, 2025","7 febrero, 2025",ES,82.22,80,80,80,80,100,...,Costa Norte Ponta Das Canas,Florianópolis,Brasil,4.0,Hoteles,"{'Salón de juegos','Información turística','Pi...","{'BREAKFAST','HALF_BOARD_AMERICAN_PLAN'}",149,21,Brasil - Florianópolis
23201,459123,"8 febrero, 2025","3 febrero, 2025",PT,100.00,100,100,100,100,100,...,Coroa Vermelha Beach All Inclusive,Porto Seguro,Brasil,3.0,Hoteles,"{'TV en zonas comunes','Servicio de guarda-equ...",NaN,32,47,Brasil - Porto Seguro
23202,374176,"8 febrero, 2025","2 febrero, 2025",ES,91.11,100,80,80,100,100,...,Piazza Hotel,Villa Carlos Paz,Argentina,3.0,Hoteles,"{'Propiedad libre de humo','TV en zonas comune...",{'BREAKFAST'},48,71,Argentina - Villa Carlos Paz
23203,5292495,"8 febrero, 2025","7 febrero, 2025",ES,75.56,100,60,60,80,100,...,Hotel Hernandez CTG,Cartagena De Indias,Colombia,3.0,Hoteles,"{'Gimnasio','Wi-Fi gratis en zonas comunes','M...",NaN,185,57,Colombia - Cartagena De Indias
23204,925087,"8 febrero, 2025","2 febrero, 2025",ES,100.00,100,100,100,100,100,...,Prodigy Gramado by Wish,Gramado,Brasil,4.0,Hoteles,"{'Cámaras de seguridad en zonas comunes','Asce...",{'BUFFET_BREAKFAST'},34,1,Brasil - Gramado


### Reprocesamiento De Reseñas Faltantes

Luego de identificar las filas que no fueron incluidas en el resultado principal, ejecutamos una segunda pasada únicamente sobre ese subconjunto.

En esta etapa usamos batches más pequeños, bajo la premisa de que algunas de estas reviews pudieron haber sido las que generaron errores, timeouts o respuestas inválidas durante el procesamiento masivo. Al reducir el tamaño del batch, disminuimos la carga por llamada y aumentamos la probabilidad de recuperar correctamente estos casos pendientes sin tener que reprocesar todo el dataset.

In [ ]:
# Definimos un tamaño de batch más pequeño para reprocesar las filas faltantes
BATCH_SIZE = 25

# Definimos cada cuántos batches vamos a guardar un checkpoint parcial
CHECKPOINT_EVERY = 10  # guarda cada 10 batches

# Inicializamos la lista donde vamos a acumular los resultados reprocesados
resultados_faltantes = []

# Inicializamos el contador de filas procesadas para esta segunda pasada
filas_procesadas = 0

# Inicializamos el contador de batches fallidos para esta segunda pasada
failed_batches_count = 0

In [ ]:
# Recorremos únicamente el DataFrame de filas faltantes en batches más pequeños
for batch_num, i in enumerate(
    tqdm(
        range(
            filas_procesadas,
            len(df_faltantes_full),
            BATCH_SIZE
        ),
        desc="Procesando batches"
    ),
    start=(filas_procesadas // BATCH_SIZE) + 1 + failed_batches_count
):

    # Tomamos el batch actual de filas faltantes
    batch_df = df_faltantes_full.iloc[i:i+BATCH_SIZE]

    # Convertimos el batch a lista de diccionarios para enviarlo al modelo
    rows = batch_df.to_dict("records")

    # Ejecutamos la llamada al modelo usando la función con retries y timeout
    results = await run_with_retry(
        rows,
        batch_num
    )

    # Si el batch falla definitivamente, lo salteamos y continuamos con el siguiente
    if results is None:
        continue
    
    # Guardamos las respuestas crudas del modelo
    outputs = []

    for res in results:

        # Extraemos el contenido textual de cada respuesta
        text = res.choices[0].message.content
        outputs.append(text)

    # Parseamos las respuestas del modelo a JSON de forma robusta
    parsed = []

    for o in outputs:

        try:

            # Intentamos convertir cada respuesta en un diccionario
            parsed.append(json.loads(o))

        except Exception as e:

            # Si una respuesta no es un JSON válido, registramos el error
            print(f"Error parseando JSON: {e}")

            # Agregamos un diccionario vacío para no cortar el reprocesamiento
            parsed.append({})

    # Construimos las filas estructuradas a partir de las reviews faltantes y sus sentimientos
    for row, sentiments in zip(rows, parsed):

        # Creamos la fila base con identificador y comentarios originales
        fila = {
            "hotel_id_review": row["hotel_id_review"],
            "Comentarios": (
                f"Comentario pos={row['texto_positivo']}. "
                f"Comentario neg={row['texto_negativo']}"
            )
        }

        # Recorremos todas las categorías definidas en el YAML
        for categoria in categorias:

            nombre = categoria["nombre"]

            # Recuperamos el valor asignado por el modelo para la categoría
            valor = sentiments.get(nombre, 0)

            try:

                # Solo aceptamos valores int, float o string para poder convertirlos a entero
                if not isinstance(valor, (int, float, str)):
                    raise ValueError(
                        f"Tipo inválido recibido: {type(valor)}"
                    )

                # Guardamos el sentimiento como entero: 1, -1 o 0
                fila[nombre] = int(valor)

            except Exception as e:

                # Si el valor no se puede convertir correctamente, registramos el problema
                print(
                    f"Error parseando categoria '{nombre}' "
                    f"en batch {batch_num}: {e}"
                )

                # Guardamos el output problemático para revisarlo después
                with open(
                    "bad_outputs.txt",
                    "a",
                    encoding="utf-8"
                ) as f:

                    f.write(
                        f"\n---\n"
                        f"Batch: {batch_num}\n"
                        f"Categoria: {nombre}\n"
                        f"Valor recibido: {valor}\n"
                        f"Output completo:\n"
                        f"{json.dumps(sentiments, ensure_ascii=False)}\n"
                    )

                # Asignamos 0 como fallback para no interrumpir el reprocesamiento
                fila[nombre] = 0

        # Agregamos la fila reprocesada al acumulado de faltantes
        resultados_faltantes.append(fila)

    # Mostramos el avance del reprocesamiento de faltantes
    print(
        f"Procesados: {i + len(batch_df)} / {len(df_faltantes_full)}"
    )

    # Guardamos un checkpoint incremental cada cierta cantidad de batches
    if batch_num % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(resultados_faltantes)

        checkpoint_df.to_csv(
            CHECKPOINT,
            index=False
        )

        print(
            f"Checkpoint guardado en batch {batch_num}"
        )

# Convertimos los resultados faltantes reprocesados en un DataFrame final
df_resultados_faltantes = pd.DataFrame(resultados_faltantes)

A continuación, combinamos los primeros resultados con los faltantes.

In [ ]:
# Unimos los resultados del procesamiento principal con los resultados reprocesados
df_final = pd.concat(
    [
        resultados_pre,
        df_resultados_faltantes
    ],
    ignore_index = True
)


# Guardamos el dataset final de reviews estructuradas
df_final.to_csv(
    OUTPUT_PROCESAMIENTO_REVIEWS,
    index = False
)

# Mostramos la cantidad final de filas guardadas
print(
    f"CSV guardado con {len(df_final)} filas"
)

CSV guardado con 251324 filas


### Agregación De Sentimiento Por Hotel

Por último, una vez consolidado el dataset final de reseñas procesadas, calculamos un score agregado por categoría a nivel hotel. Para ello agrupamos todas las reviews utilizando el identificador del hotel (hotel_id) y computamos métricas agregadas sobre cada una de las categorías de sentimiento generadas previamente por el modelo.

Durante esta etapa se excluyen las columnas textuales (como Comentarios) y se trabajan únicamente las columnas numéricas de categorías (wifi, limpieza, ubicacion, personal, etc.). De esta manera obtenemos una representación resumida del sentimiento de los huéspedes para cada aspecto relevante del hotel, permitiendo construir perfiles comparables entre hoteles y facilitando posteriores análisis, rankings o visualizaciones.

In [ ]:
# Cargamos el dataset final de reviews estructuradas
df_final = pd.read_csv(
    OUTPUT_PROCESAMIENTO_REVIEWS
)

# Definimos las columnas que no vamos a incluir en la agregación numérica
exclude_cols = [
    "hotel_id_review",
    "Comentarios"
]

# Identificamos las columnas de categorías generadas por el modelo
category_cols = [
    c for c in df_final.columns
    if c not in exclude_cols
]

# Agregamos los sentimientos por hotel usando suma
# Esto nos da un score acumulado por categoría para cada hotel
df_hotel_scores = (
    df_final
    .groupby("hotel_id_review")[category_cols]
    .sum()
    .reset_index()
)


# Creamos el campo destino en el dataset original
df_work["destino"] = (
    df_work["country"].astype(str)
    + " - "
    + df_work["city"].astype(str)
)

# Nos quedamos con una sola fila por hotel para asociar cada hotel con su destino
df_destinos = (
    df_work[
        ["hotel_id_review", "destino"]
    ]
    .drop_duplicates(
        subset = ["hotel_id_review"]
    )
)

# Incorporamos el destino a la tabla de scores agregados por hotel
df_hotel_scores = df_hotel_scores.merge(
    df_destinos,
    on = "hotel_id_review",
    how = "left"
)


# Guardamos el dataset final de scores agregados por hotel
df_hotel_scores.to_csv(
    OUTPUT_HOTEL_SCORES,
    index = False,
    encoding = "utf-8-sig"
)

### Construcción De Base SQL Para Consulta De Hoteles

Una vez calculados los scores agregados por hotel, construimos una base de datos SQL para facilitar la consulta de esta información.

En esta etapa cargamos el dataset de scores por hotel y lo guardamos como una tabla llamada `hoteles` dentro de una base SQLite. Esto nos permite consultar los hoteles de forma estructurada, filtrando por destino o comparando scores entre categorías como limpieza, wifi, ubicación, personal, desayuno, entre otras.

La idea es que esta tabla funcione como una capa consultable sobre los resultados del procesamiento anterior. En lugar de recorrer manualmente el DataFrame, podemos formular consultas SQL para recuperar hoteles relevantes según distintos criterios.

Luego, sobre esta base, usamos un LLM para transformar preguntas escritas en lenguaje natural en queries SQL. De esta forma, el usuario puede pedir algo como “hoteles en Río de Janeiro con buena limpieza y buena ubicación”, y el modelo puede traducir esa intención en una consulta sobre la tabla `hoteles`.

In [ ]:
# Cargamos el dataset de scores agregados por hotel
df_hoteles = pd.read_csv(OUTPUT_HOTEL_SCORES)

# Creamos la base de datos SQLite y establecemos la conexión
conn = sqlite3.connect(BASE_DE_DATOS)

# Guardamos el DataFrame como una tabla SQL llamada "hoteles"
df_hoteles.to_sql(
    name = "hoteles", # Nombre de la tabla
    con = conn, # Conexión a la base SQLite
    if_exists = "replace", # Reemplazamos la tabla si ya existe
    index = False # No guardamos el índice del DataFrame
)

2523

Luego configuramos nuevamente el cliente compatible con OpenAI, apuntando al servidor local de vLLM.

En esta etapa vamos a usar el modelo para una tarea distinta: en lugar de estructurar sentimientos desde reviews, le vamos a pedir que genere consultas SQL a partir de preguntas en lenguaje natural. Para eso definimos una función asincrónica que recibe una consulta del usuario y un system prompt específico para guiar la generación de SQL.

In [ ]:
# Creamos nuevamente el cliente asincrónico apuntando al servidor local de vLLM
client = AsyncOpenAI(
    base_url = "http://localhost:8000/v1",
    api_key = "dummy"
)

# Definimos una función para enviar consultas al modelo
async def model_query(query, sytem_promp):

    return await client.chat.completions.create(
        model = "Qwen/Qwen2.5-3B-Instruct",
        messages = [
            # Pasamos el system prompt que define cómo debe comportarse el modelo
            {
                "role": "system",
                "content": sytem_promp
            },
            # Pasamos la consulta del usuario en lenguaje natural
            {
                "role": "user",
                "content": query
            }
        ],
        # Usamos temperatura 0 para obtener respuestas más consistentes
        temperature = 0
    )

### Armado Del Prompt

Ahora debemos escribir un system prompt para un asistente especializado en generación de queries SQL para hoteles. El objetivo es que el modelo interprete correctamente las necesidades del usuario, identifique categorías IMPORTANTES y NO IMPORTANTES, y construya una query SQL válida siguiendo reglas estrictas de combinación lógica (AND / OR).

El modelo debe distinguir entre:

- Requerimientos fundamentales u obligatorios del usuario
- Preferencias deseables pero no excluyentes

Las categorías IMPORTANTES representan condiciones que el usuario considera necesarias o indispensables y deben cumplirse obligatoriamente.

Las categorías NO IMPORTANTES representan preferencias opcionales, agradables o secundarias, que aumentan la relevancia del hotel pero no son excluyentes.

La clasificación debe inferirse tanto por:

- Palabras explícitas de prioridad
- Intensidad del lenguaje
- Contexto semántico de la frase

Además, la salida debe devolverse exclusivamente en formato JSON con la query SQL generada y nada más.

In [ ]:
# Construimos el texto descriptivo de categorías a partir del YAML
categorias_texto = "\n".join(
    [
        f'- {cat["nombre"]}: score de {cat["descripcion"]}'
        for cat in categorias
    ]
)

# Construimos el prompt que va a guiar al modelo para generar queries SQL
QUERY_GENERATOR_PROMPT = f"""
Sos un generador de queries SQL. 
Tu tarea es entender las necesidades hoteleras del usuario y traducirlas a una query SQL válida.

Debés devolver SOLO un JSON válido con esta estructura:

{{
  "query": "..."
}}

No agregues explicaciones, markdown ni texto adicional.

Nombre de la tabla: hoteles

Columnas disponibles:
- hotel_id_review: identificador del hotel
- destino: lugar donde se encuentra el hotel
{categorias_texto}

Template SQL obligatorio:

SELECT hotel_id_review, destino, <<categorias mencionadas por el usuario>>
FROM hoteles
WHERE destino = "<<destino mencionado por el usuario>>"
AND <<condiciones en base a categorias>>

Reglas para definir las condiciones por categoría:

1) Identificá todas las categorías relevantes mencionadas por el usuario.

2) Clasificá las categorías en dos grupos:

Categorías IMPORTANTES:
Son aquellas donde el usuario expresa alta prioridad o necesidad fundamental.

Ejemplos:
- "fundamentalmente"
- "es clave"
- "muy importante"
- "necesito sí o sí"
- "indispensable"

Estas categorías deben agregarse usando:

categoria > 0

y deben combinarse entre sí usando AND.

Categorías NO IMPORTANTES:
Son preferencias deseables, pero no excluyentes.

Ejemplos:
- "me gustaría"
- "con buena vista"
- "ojalá tenga"
- "preferentemente"
- "también estaría bueno"

Estas categorías deben agregarse usando:

categoria > 0

y deben combinarse entre sí usando OR dentro de un único bloque.

3) Si existen categorías IMPORTANTES y NO IMPORTANTES al mismo tiempo:

- Todas las IMPORTANTES deben cumplirse obligatoriamente usando AND.
- Todas las NO IMPORTANTES deben agruparse en un único bloque OR.
- Ese bloque OR debe conectarse con las IMPORTANTES usando AND.

Ejemplo de condición:

AND limpieza > 0
AND (vista > 0 OR desayuno > 0)

4) Si solo existen categorías NO IMPORTANTES:

Combiná todas usando OR dentro de un único bloque.

Ejemplo:

AND (vista > 0 OR desayuno > 0)

5) Si solo existen categorías IMPORTANTES:

Combiná todas usando AND.

Ejemplo:

AND limpieza > 0
AND ubicacion > 0

Reglas importantes:

- Las categorías NO IMPORTANTES nunca deben combinarse con AND entre sí.
- Todas las categorías NO IMPORTANTES deben agruparse dentro de un único bloque OR.
- No agregues categorías que el usuario no haya mencionado.
- No inventes destinos.
- No inventes columnas.
- Usá solamente columnas incluidas en la lista de columnas disponibles.
- Devolvé únicamente el JSON final.

Ejemplo:

Usuario:
Quiero un hotel en Brasil - Rio De Janeiro con buena vista y buen desayuno. Es fundamental una buena ubicacion.

Respuesta esperada:
{{
  "query": "SELECT hotel_id_review, destino, ubicacion, vista, desayuno FROM hoteles WHERE destino = 'Brasil - Rio De Janeiro' AND ubicacion > 0 AND (vista > 0 OR desayuno > 0)"
}}
"""

# Imprimimos el prompt para revisar que se haya construido correctamente
print(QUERY_GENERATOR_PROMPT)

### Cálculo Del Score Final Por Hotel

Luego de ejecutar la query SQL y obtener los hoteles candidatos, se calcula un score final para rankear los resultados de forma homogénea entre categorías.

##### Paso 1: eliminación de valores negativos

Todas las columnas de categorías (wifi, vista, limpieza, etc.) son transformadas de la siguiente manera:

```python
valor = max(valor, 0)
```

Es decir:
- Cualquier valor negativo se reemplaza por `0`
- Esto evita penalizaciones extremas
- Los scores negativos se consideran ausencia de señal positiva


##### Paso 2: normalización por categoría

Cada categoría es normalizada utilizando el valor máximo encontrado en esa columna.

Fórmula:

```text
valor_normalizado = valor / maximo_columna
```

De esta manera:
- Todas las categorías quedan en la misma escala `[0,1]`
- Ninguna categoría domina el ranking únicamente por tener números más grandes

##### Paso 3: cálculo del score final

El score final del hotel se calcula como la suma de todas las categorías normalizadas:

```text
hotel_score =
wifi +
vista +
ubicacion +
limpieza +
personal +
...
```

Esto genera un ranking global donde:
- Scores altos representan hoteles consistentemente buenos en múltiples dimensiones
- Scores bajos representan hoteles con pocas fortalezas relevantes

##### Ventajas del enfoque

- Simple de interpretar
- Comparable entre categorías heterogéneas
- Evita sesgos por escalas distintas
- Permite ordenar hoteles de manera robusta y transparente

In [ ]:
def score_hotel(resultado):

    # Definimos las columnas que no vamos a usar para calcular el score
    cols_excluir = ["hotel_id_review", "destino"]

    # Identificamos las columnas numéricas de categorías a procesar
    score_cols = [
        c for c in resultado.columns
        if c not in cols_excluir
    ]

    # Creamos una copia para no modificar el DataFrame original
    df_scores = resultado.copy()

    # Llevamos los valores negativos a 0
    # En esta etapa priorizamos señales positivas para el ranking final
    df_scores[score_cols] = df_scores[score_cols].clip(lower = 0)

    # Normalizamos cada categoría por su valor máximo
    for col in score_cols:

        max_val = df_scores[col].max()

        # Evitamos división por cero cuando una categoría no tiene valores positivos
        if max_val > 0:
            df_scores[col] = df_scores[col] / max_val
        else:
            df_scores[col] = 0

    # Calculamos el score final sumando las categorías normalizadas
    df_scores["hotel_score"] = df_scores[score_cols].sum(axis = 1)

    # Ordenamos los hoteles de mayor a menor score final
    df_scores = df_scores.sort_values(
        "hotel_score",
        ascending = False
    )

    return df_scores

In [ ]:
# Prompt ficticio para probar el módulo
consulta_usuario = "Recomendame un hotel en ¨Argentina - El Calafate¨, fundamentalmente con una buena vista. Estaria bueno si tambien tiene buena limpieza y wifi"

In [ ]:
# Resultados del módulo
conn = sqlite3.connect(BASE_DE_DATOS)
query = await model_query(consulta_usuario,QUERY_GENERATOR_PROMPT)
query = json.loads(query.choices[0].message.content.strip()).get("query", 0)
print("Query Generada:", query)
resultado = pd.read_sql_query(query, conn)
score_hotel(resultado).head()

Query Generada: Select hotel_id_review, destino, vista, limpieza, wifi from hoteles where destino = 'Argentina - El Calafate' AND vista > 0 AND (limpieza > 0 OR wifi > 0)


,hotel_id_review,destino,vista,limpieza,wifi,hotel_score
14,846786,Argentina - El Calafate,1.000000,0.451613,0.0,1.451613
6,325052,Argentina - El Calafate,0.153846,0.193548,1.0,1.347395
12,481901,Argentina - El Calafate,0.523077,0.774194,0.0,1.297270
5,325045,Argentina - El Calafate,0.292308,1.000000,0.0,1.292308
4,325030,Argentina - El Calafate,0.307692,0.548387,0.0,0.856079
